
# YOLO26 Inference
This notebook runs detection using Ultralytics YOLO26.
It does not train models. The first run will download weights.

**Note:** If YOLO26 is not available in the current Ultralytics package version, it will fail clearly as requested.


In [ ]:

import sys
from pathlib import Path

import cv2
import yaml

sys.path.append(str(Path.cwd().parent))
import torch

from src.aeronetra.counting.drawing import (
    draw_count_summary,
    draw_detections,
    export_to_json,
)
from src.aeronetra.counting.ops import count_vehicles
from src.aeronetra.detection.adapters import get_model_adapter
from src.aeronetra.detection.types import CountSummary


In [ ]:

# Configuration
config_path = Path("../configs/inference/inference.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "yolo26n.pt" # Assuming 'n' for nano. Adjust if exact API name differs
weights_path = model_name
class_names = {0: "person", 2: "car", 5: "bus", 7: "truck"} # COCO subset example
selected_classes = [2, 5, 7] # Vehicles


In [ ]:

# Load Model explicitly
try:
    print(f"Attempting to load {model_name}...")
    adapter = get_model_adapter("YOLO26", weights_path, class_names, device)
    adapter.load_model()
    print("Model loaded successfully.")
except (ImportError, RuntimeError) as e:
    print("Failed to load YOLO26. Ensure you have the correct Ultralytics version supporting YOLO26.")
    print(f"Error: {e}")
    adapter = None


In [ ]:

# Inference on sample image
if adapter is not None:
    sample_img_path = Path("../tests/fixtures/sample.jpg")
    if sample_img_path.exists():
        img = cv2.imread(str(sample_img_path))

        # Inference
        prediction = adapter.predict(img, conf_thresh=config["confidence_threshold"], iou_thresh=config["iou_threshold"])
        print(f"Inference time: {prediction.inference_time_ms:.2f} ms")

        # Filter
        prediction.filter_by_class(selected_classes)
        print(f"Detections after class filtering: {len(prediction.detections)}")

        # Count
        total, c_counts = count_vehicles(prediction.detections)
        summary = CountSummary("sample.jpg", total, c_counts, "YOLO26")

        # Draw and Save
        out_img = draw_detections(img, prediction.detections)
        out_img = draw_count_summary(out_img, summary)

        out_dir = Path("../outputs/predictions")
        out_dir.mkdir(parents=True, exist_ok=True)

        cv2.imwrite(str(out_dir / "yolo26_out.jpg"), out_img)
        export_to_json(prediction.detections, out_dir / "yolo26_preds.json")
        print("Outputs saved.")
    else:
        print(f"Sample image {sample_img_path} not found. Please add a fixture image.")
